# 3. Compare salary with performance

This notebook joins the 2022 salary file to that season's player stats, calculates an availability-adjusted score, and compares the score with salary. The result is a simple screening measure; it does not account for contract rules or establish fair value. Run it from the repository root.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression

DATA_DIR = Path("data")
salary_df = pd.read_csv(DATA_DIR / "raw" / "salaries" / "nba_salaries.csv")
stats_df = pd.read_csv(DATA_DIR / "raw" / "statistics" / "player_stats_2022.csv")

print(f"Salary rows: {len(salary_df):,}")
print(f"Stats rows: {len(stats_df):,}")
salary_df.head()

## Join by player name

The source files use different names for the salary fields, so rename them before merging.

In [ ]:
salary_df = salary_df.rename(
    columns={"Player Name": "PLAYER_NAME", "Salary": "SALARY"}
)

merged_df = stats_df.merge(
    salary_df[["PLAYER_NAME", "SALARY"]],
    on="PLAYER_NAME",
    how="inner",
)

print(f"Matched {len(merged_df):,} player rows")
merged_df[["PLAYER_NAME", "SALARY"]].sample(10, random_state=7)

## Save the joined rows

Keep a copy of the matched salary and stats before calculating the comparison scores.

In [ ]:
merged_path = DATA_DIR / "merged_nba_salary_stats_2022.csv"
merged_df.to_csv(merged_path, index=False)
print(f"Saved {len(merged_df):,} rows to {merged_path}")

## Adjust for games played

This score includes points, rebounds, assists, steals, blocks, and turnovers. Dividing by salary in millions makes it easier to compare pay and production, but the score is not a full measure of player value.

In [ ]:
merged_df["performance_score"] = (
    merged_df["PTS"]
    + 1.2 * merged_df["REB"]
    + 1.5 * merged_df["AST"]
    + 3 * merged_df["STL"]
    + 3 * merged_df["BLK"]
    - 1.5 * merged_df["TOV"]
)
merged_df["availability"] = merged_df["GP"] / 82
merged_df["risk_adj_performance"] = merged_df["performance_score"] * merged_df["availability"]
merged_df["value_per_million"] = merged_df["risk_adj_performance"] / (merged_df["SALARY"] / 1_000_000)

merged_df[["PLAYER_NAME", "SALARY", "risk_adj_performance", "value_per_million"]].sort_values(
    "value_per_million", ascending=False
).head(20)

## Compare players with a basic filter

Filtering out players with fewer than 40 games or less than $1.5 million in salary gives a more consistent comparison set.

In [ ]:
analysis_df = merged_df[
    (merged_df["GP"] >= 40) & (merged_df["SALARY"] >= 1_500_000)
].copy()

analysis_df[["PLAYER_NAME", "SALARY", "GP", "risk_adj_performance", "value_per_million"]].sort_values(
    "value_per_million", ascending=False
).head(20)

## Compare pay with a simple regression

This line predicts the performance score from salary alone. The difference between a player's score and the fitted value is a model residual, not proof that a contract is good or bad.

In [ ]:
X = analysis_df[["SALARY"]]
y = analysis_df["risk_adj_performance"]
model = LinearRegression().fit(X, y)
analysis_df["expected_performance"] = model.predict(X)
analysis_df["alpha"] = analysis_df["risk_adj_performance"] - analysis_df["expected_performance"]

analysis_df[["PLAYER_NAME", "SALARY", "risk_adj_performance", "expected_performance", "alpha"]].sort_values(
    "alpha", ascending=False
).head(20)

In [ ]:
analysis_df[["PLAYER_NAME", "SALARY", "risk_adj_performance", "expected_performance", "alpha"]].sort_values(
    "alpha", ascending=True
).head(20)

## Save the joined analysis

In [ ]:
output_path = DATA_DIR / "nba_contract_valuation_2022.csv"
analysis_df.to_csv(output_path, index=False)
print(f"Saved {len(analysis_df):,} rows to {output_path}")